# DNA as a Language — Guided K-mer Analysis Exercises
**AMLG | FASTA files, k-mer counting, k-mer spectra, and sequence similarity**

This notebook is designed for a live teaching session **and independent practice**. You will turn DNA sequences into short overlapping words (*k-mers*), count their frequencies, and compare homologous genes from several species.

### Learning goals
- Read DNA sequences from FASTA files and organize them in Python dictionaries.
- Extract overlapping k-mers and count how often each appears.
- Interpret a k-mer frequency spectrum from an *E. coli* reference genome.
- Compute Jaccard similarity for two k-mer **sets**.
- Compare homologous genes across species with a similarity matrix.

### How to use this notebook
1. Run cells **top to bottom** with **Shift + Enter**. Work from your repository root as in the original exercise.
2. Before each ❓ exercise, read **Task**, **Hints**, **What this teaches**, and **Expected outcome**.
3. Fill in the `# TODO` / `# YOUR CODE` positions. Starters are deliberately incomplete.
4. The commented solution is below `# --- Solution (uncomment to check) ---`. Try independently before revealing it.
5. Remove `pass` where relevant, uncomment the solution lines if needed, and rerun the exercise and subsequent cells.
6. Inspect outputs and use the quick checks to verify your implementation.

**Learning route:** FASTA reading → k-mer extraction and counting → k-mer spectrum → Jaccard index → homologous-gene comparisons → cross-species heatmaps.

> **Repository note:** Original relative file paths, image references, and download URLs are retained. Run the notebook in the same repository environment as the provided exercise. The setup cells are preserved rather than replaced with new paths.

> **Exercise convention:** Cells with TODOs may fail or show incomplete outputs until you implement them. This is intentional: work through the TODOs before running later dependent cells.

# **DNA as a Language: K-Mers for Sequence Analysis**

## 0. Data and environment
We work with **FASTA** files. A FASTA entry starts with a header line beginning with `>`, followed by one or more lines containing a DNA sequence. A file may contain one or many entries.

The original setup below downloads files into `data/genomes/` and `data/homologous-genes/`. Those locations are used throughout the notebook. If the data are already present in your repository, use the original setup according to your course instructions; no file paths are changed here.

**Python recap:** a *dictionary* maps a key (e.g., gene description) to a value (e.g., DNA sequence); a *list* keeps an ordered series of items; a *set* contains distinct items only.

First, let's download the data for this notebook.

In [ ]:
import os
import sys

# Append the root of the Git repository to the path.
git_root = os.popen(cmd="git rev-parse --show-toplevel").read().strip()
sys.path.append(git_root)

In [ ]:
from utils import download_file

if not os.path.exists(path="data"):
    os.makedirs(name="data/genomes")
    os.makedirs(name="data/homologous-genes")

download_file(
    # url="https://dataverse.harvard.edu/api/access/datafile/10494340",
    url="https://seafile.cloud.uni-hannover.de/d/5d6029c6eaaf410c8b01/files/?p=%2Fkmer_analysis%2Fgenomes%2Fe-coli-dh10b.fasta&dl=1",
    save_filename="data/genomes/e-coli-dh10b.fasta",
)
download_file(
    # url="https://dataverse.harvard.edu/api/access/datafile/10494341",
    url="https://seafile.cloud.uni-hannover.de/d/5d6029c6eaaf410c8b01/files/?p=%2Fkmer_analysis%2Fhomologous_genes%2Fc_elegans.fasta&dl=1",
    save_filename="data/homologous-genes/c-elegans.fasta",
)
download_file(
    # url="https://dataverse.harvard.edu/api/access/datafile/10494348",
    url="https://seafile.cloud.uni-hannover.de/d/5d6029c6eaaf410c8b01/files/?p=%2Fkmer_analysis%2Fhomologous_genes%2Fd_melanogaster.fasta&dl=1",
    save_filename="data/homologous-genes/d-melanogaster.fasta",
)
download_file(
    # url="https://dataverse.harvard.edu/api/access/datafile/10494347",
    url="https://seafile.cloud.uni-hannover.de/d/5d6029c6eaaf410c8b01/files/?p=%2Fkmer_analysis%2Fhomologous_genes%2Fh_sapiens.fasta&dl=1",
    save_filename="data/homologous-genes/h-sapiens.fasta",
)
download_file(
    # url="https://dataverse.harvard.edu/api/access/datafile/10494342",
    url="https://seafile.cloud.uni-hannover.de/d/5d6029c6eaaf410c8b01/files/?p=%2Fkmer_analysis%2Fhomologous_genes%2Fm_musculus.fasta&dl=1",
    save_filename="data/homologous-genes/m-musculus.fasta",
)
download_file(
    # url="https://dataverse.harvard.edu/api/access/datafile/10494345",
    url="https://seafile.cloud.uni-hannover.de/d/5d6029c6eaaf410c8b01/files/?p=%2Fkmer_analysis%2Fhomologous_genes%2Fp_troglodytes.fasta&dl=1",
    save_filename="data/homologous-genes/p-troglodytes.fasta",
)

In [ ]:
from typing import Dict, List

## 1. Explore homologous genes
**Concept:** Homologous genes share evolutionary ancestry. Here, corresponding FASTA entries across species represent homologous genes, enabling a meaningful sequence comparison.

**Reading check:** In the given `read_fasta_file` function, what happens when a line starts with `>`? Why is the dictionary key a sequence description rather than a sequence index?

# Homologous Genes

The `data/` folder contains two subfolders: `genomes/` and `homologous-genes/`.
The `genomes/` folder contains the reference genome of _E. coli_ strain DH10B as FASTA file `e-coli-dh10b.fasta`.
The `homologous-genes/` folder contains 5 FASTA files.
Each file contains sequences of one specific species:

- [Human](https://en.wikipedia.org/wiki/Human) (_Homo sapiens_): `h-sapiens.fasta`
- [Chimpanzee](https://en.wikipedia.org/wiki/Chimpanzee) (_Pan troglodytes_): `p-troglodytes.fasta`
- [House mouse](https://en.wikipedia.org/wiki/House_mouse) (_Mus musculus_): `m-musculus.fasta`
- [Threadworm](https://en.wikipedia.org/wiki/Caenorhabditis_elegans) (_Caenorhabditis elegans_): `c-elegans.fasta`
- [Fruit fly](https://en.wikipedia.org/wiki/Drosophila_melanogaster) (_Drosophila melanogaster_): `d-melanogaster.fasta`

The i-th sequences in the files are homologuous sequences of the same gene in the different species (i.e., they are similar due to shared ancestry).

The following function can be used to read a FASTA file into a dictionary.
In the dictionary, the sequence descriptions are the keys, and the actual sequences are the values.

In [ ]:
def read_fasta_file(file_path: str) -> Dict[str, str]:
    """Read a FASTA file and return the sequences as a dictionary."""
    sequences = {}

    with open(file=file_path, mode="r") as file:
        current_description = None
        for line in file:
            line = line.strip()
            if line.startswith(">"):
                current_description = line[1:]
                sequences[current_description] = ""
            else:
                sequences[current_description] += line

    return sequences

### ❓ Exercise 1 — Discover and read multiple FASTA files
**Task:** Finish `list_files_in_directory()` so it returns the paths of the files found by `os.walk`. Then use the existing loop to load each file with `read_fasta_file`.

**Hints:** `os.walk` yields `(root, directories, files)`; `os.path.join(root, name)` constructs a path; use `file_list.append(...)` to build the result. You can restrict to filenames ending in `.fasta` if you want, but keep the original directory paths.

**What this teaches:** Basic file-system traversal, loops, and passing a filename to a parser.

**Expected outcome:** Five FASTA files are listed/read from `data/homologous-genes/`, with sequence-description keys printed for each. File order may vary.

**Quick check:** `len(list_files_in_directory('data/homologous-genes/'))` should be **5** in the provided course dataset.

##### ❓ Reading in multiple FASTA files

Complete the following code to read in the 5 provided FASTA files from the `data/homologous-genes/` folder.

In [ ]:
import os


def list_files_in_directory(directory: str) -> List[str]:
    """List all files in the specified directory."""
    file_list = []

    # Iterate over the files in the specified directory.
    for root, _, files in os.walk(top=directory):
        # TODO: append a full path for each file to file_list.
        pass  # Remove after implementing the TODO.

        # --- Solution (uncomment to check) ---
        # for filename in files:
        #     file_list.append(os.path.join(root, filename))

    return file_list


# Print the list of files.
for file in list_files_in_directory(directory="data/homologous-genes/"):
    sequences = read_fasta_file(file_path=file)
    print(f"Read {len(sequences)} sequences from '{file}': {list(sequences.keys())}")


## 2. From DNA sequences to k-mers
A **k-mer** is a sequence window of exactly `k` characters. Slide the window by one position to obtain overlapping k-mers.

**Worked example** — `GATTACA`, `k = 3`:

| Start index | k-mer |
|---:|:---|
| 0 | `GAT` |
| 1 | `ATT` |
| 2 | `TTA` |
| 3 | `TAC` |
| 4 | `ACA` |

If the sequence length is `L` and `1 ≤ k ≤ L`, the number of overlapping windows is `L - k + 1` (including repetitions). The number of **distinct** k-mers may be smaller. For DNA made from `A, C, G, T`, there are at most `4**k` possible words.

**Think about it:** A larger `k` generally makes an exact match more informative, but also makes it easier for one mutation to break a match.

# K-Mers

_k_-mers are substrings of length $k$ contained within a biological sequence.
They are a fairly simple concept that turns out to be tremendously powerful.

Usually, the term _k_-mer refers to all of a sequence's subsequences of length $k$.
Using this definition, the sequence `ACGT` would have four monomers (`A`, `C`, `G`, `T`), three 2-mers (`AC`, `CG`, `GT`), two 3-mers (`ACG`, `CGT`), and one 4-mer (`ACGT`).
More generally, a sequence of length $L$ has $L - k + 1$ _k_-mers and $n^k$ total possible _k_-mers, where $n$ is the number of possible monomers.

Typically we extract _k_-mers from genomic assemblies or read data sets by running a $k$-length window across all of the sequences.
E.g., given the sequence `AGGATGAGACAGATAG` of length 16, you could extract the following 6-mers from it:

```
AGGATG
 GGATGA
  GATGAG
   ATGAGA
    TGAGAC
     GAGACA
      AGACAG
       GACAGA
        ACAGAT
         CAGATA
          AGATAG
```

_k_-mers are used in numerous applications, e.g.:

- The frequency of a set of _k_-mers in a species's genome, in a genomic region, or in a class of sequences can be used as a "signature" of the underlying sequence.
- With respect to disease, dinucleotide bias has been applied to the detection of genetic islands associated with pathogenicity.
- _k_-mer frequency and spectrum variation is heavily used in metagenomics analysis.

_k_-mers are most useful when they are long, because then they are specific.
That is, if you have a 31-mer taken from a human genome, it is pretty unlikely that another genome has that exact 31-mer in it.

> You can calculate the probability if you assume genomes are random: there are $n^k = 4^{31} = 4,611,686,018,427,387,904$ possible 31-mers&nbsp;&mdash;&nbsp;a lot.
> The human genome has a size of $L = 3,117,275,501$.
> We can hence extract $L - k + 1 = 3,117,275,501 - 31 + 1 = 3,117,275,471$ 31-mers from it.
> The probability that a random genome contains a specific 31-mer is hence $\frac{3,117,275,471}{4,611,686,018,427,387,904} \approx 0.0000000006$.
>
> An extension of this observation is that long _k_-mers are species specific.

### ❓ Exercise 2 — Build and count k-mers
**Task:** Implement `build_kmers(sequence, k)` to return (1) all overlapping windows in order and (2) a dictionary mapping each distinct k-mer to its occurrence count.

**Hints:** Iterate over `range(len(sequence) - k + 1)`, slice with `sequence[i:i+k]`, then update a dictionary using `.get(word, 0) + 1`. The last window must be included.

**What this teaches:** String slicing, sliding windows, loops, and frequency dictionaries.

**Expected outcome:** For `GATTACACA` with `k=3`, the list has **7** entries and `ACA` appears **twice**. All counts should sum to 7.

**Optional edge-case check:** Decide how you want to handle `k < 1` or `k > len(sequence)`; the suggested solution raises a `ValueError` for invalid `k`.

##### ❓ Building _k_-mers

Complete the function `build_kmers()`.
The function shall do two things:

1. Build a list of _k_-mers for the given `sequence`.
2. Count the frequency of each _k_-mer.

As an example, for the `sequence = "GATTACACA"` the function shall return `kmers = ['GAT', 'ATT', 'TTA', 'TAC', 'ACA', 'CAC', 'ACA']` and `kmer_counts = {'GAT': 1, 'ATT': 1, 'TTA': 1, 'TAC': 1, 'ACA': 2, 'CAC': 1}`.

In [ ]:
def build_kmers(sequence: str, k: int) -> tuple[List[str], Dict[str, int]]:
    """Build k-mers from a sequence and count the occurrences of each k-mer."""
    kmers = []
    kmer_counts = {}

    # TODO: validate k; then fill kmers and kmer_counts using a sliding window.
    pass  # Remove after implementing the TODO.

    # --- Solution (uncomment to check) ---
    # if not 1 <= k <= len(sequence):
    #     raise ValueError("k must satisfy 1 <= k <= len(sequence)")
    # for i in range(len(sequence) - k + 1):
    #     word = sequence[i : i + k]
    #     kmers.append(word)
    #     kmer_counts[word] = kmer_counts.get(word, 0) + 1

    return kmers, kmer_counts


kmers, kmer_counts = build_kmers(sequence="GATTACACA", k=3)
print(kmers)
print(kmer_counts)


**Self-check (after completing Exercise 2):** Try `sum(kmer_counts.values()) == len(kmers)` and check that the result is `True`. For `GATTACACA`, the expected list is `['GAT', 'ATT', 'TTA', 'TAC', 'ACA', 'CAC', 'ACA']` and `kmer_counts['ACA'] == 2`.

## 3. Count frequencies of frequencies: the k-mer spectrum
The `kmer_counts` dictionary answers **“How often does this particular k-mer occur?”** The **k-mer spectrum** answers **“How many distinct k-mers occur once, twice, three times, …?”**

**Mini example:** If `kmer_counts = {'AAA': 3, 'AAC': 1, 'ACC': 1}`, then two distinct k-mers have frequency 1 and one distinct k-mer has frequency 3. A spectrum plot would contain points `(1, 2)` and `(3, 1)`.

**Important distinction:** The graph counts **distinct words** at each multiplicity; it does not plot every occurrence as a separate word.

# The K-Mer Spectrum

A method of visualizing _k_-mers, the _k_-mer spectrum, shows the multiplicity of each _k_-mer in a sequence versus the number of _k_-mers with that multiplicity.
The number of modes in a _k_-mer spectrum for a species' genome varies, with most species having a unimodal distribution.
However, all mammals have a multimodal distribution.

### ❓ Exercise 3 — Plot the 7-mer spectrum of *E. coli*
**Task:** Use the `ecoli_kmer_counts` dictionary generated below to compute `kmer_frequencies` (sorted unique observed counts) and `n_kmers_with_frequency` (how many distinct words have each count). The original plot cell then visualizes the spectrum.

**Hints:** `collections.Counter(ecoli_kmer_counts.values())` counts how many k-mers share each multiplicity. Sort its keys; build corresponding y-values in the same order.

**What this teaches:** Reorganizing frequency tables and plotting an interpretable summary of a genome.

**Expected outcome:** A plot with frequency (x-axis) and number of distinct k-mers (y-axis), rather than a graph of the genome sequence. The two plotted arrays must have equal length.

**Interpretation prompt:** How many k-mers appear exactly once? What does a high-frequency k-mer suggest about repeated sequence patterns?

##### ❓ Computing a _k_-mer spectrum

Compute and plot the 7-mer spectrum of the provided _E. coli_ genome.
Here we use the DH10B strain, a widely used model organism.

> This [_E. coli_ genome](https://www.ncbi.nlm.nih.gov/nuccore/NZ_CP110018.1/) and others can be downloaded from the [NCBI](https://www.ncbi.nlm.nih.gov).

In [ ]:
ecoli_genome = read_fasta_file(file_path="data/genomes/e-coli-dh10b.fasta")
ecoli_genome_sequence = ecoli_genome[
    "NZ_CP110018.1 Escherichia coli strain DH10B chromosome, complete genome"
]
print(f"Read E. coli genome with length {len(ecoli_genome_sequence):,}.")

In [ ]:
current_k = 7
ecoli_kmers, ecoli_kmer_counts = build_kmers(
    sequence=ecoli_genome_sequence, k=current_k
)
# TODO: compute the distinct observed frequencies and the number of k-mers
#       at each frequency. The next cell plots these two lists.
# YOUR CODE

# --- Solution (uncomment to check) ---
# from collections import Counter
# frequency_histogram = Counter(ecoli_kmer_counts.values())
# kmer_frequencies = sorted(frequency_histogram)
# n_kmers_with_frequency = [frequency_histogram[f] for f in kmer_frequencies]


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 4))
plt.plot(kmer_frequencies, n_kmers_with_frequency)
plt.title(label=f"{current_k}-mer spectrum of {list(ecoli_genome.keys())[0]}")
plt.xlabel("frequency")
plt.ylabel("count")
plt.show()

## 4. Comparing two sequences without aligning them
When comparing sets of distinct k-mers, a simple metric is the **Jaccard index**:

$$J(A,B) = \frac{|A \cap B|}{|A \cup B|}.$$

- `A & B` is the intersection (shared elements).
- `A | B` is the union (all distinct elements in either set).
- `J = 1` means identical, nonempty sets; `J = 0` means no shared elements.

**Worked example:** `A={'AAA','AAC','ACC'}` and `B={'AAC','ACC','CCC'}` share 2 elements and have 4 in their union, so `J=2/4=0.5`.

**Important:** A set discards repeated k-mers. This exercise uses **presence/absence** similarity, not k-mer abundance. The original notebook defines `J(empty, empty) = 0.0`; preserve that convention in your implementation.

# The Jaccard Index

One thing we can do is to use _k_-mers to compare genomes to genomes, or read data sets to read data sets.

One metric we can use for this comparisons is the Jaccard index, which is calculated by asking how many _k_-mers are shared between two samples (expressed as the cardinality of the intersection of the two _k_-mer sets) versus how many _k_-mers in total are in the combined samples (expressed as the cardinality of the union of the two _k_-mer sets):

$$J(A,B) = \frac{|A \cap B|}{|A \cup B|}$$

<img src="images/intersection.png" width="200" />
<img src="images/union.png" width="200" />

### ❓ Exercise 4 — Implement Jaccard similarity
**Task:** Complete `jaccard_index(a, b)` using the intersection size divided by the union size. Keep the provided special case for two empty sets.

**Hints:** Use `len(a.intersection(b))` and `len(a.union(b))`, or the shorter set operators `a & b` and `a | b`.

**What this teaches:** Set operations and normalized similarity metrics.

**Expected outcome:** The existing test cell prints values between 0 and 1. For example, `J({'A'}, {'A'}) = 1.0`, `J({'A'}, {'C'}) = 0.0`, and `J({'A','C','G'}, {'C','G','T'}) = 0.5`.

**Check your reasoning:** Why is `J({'A','A'}, {'A'})` not different from `J({'A'}, {'A'})`? (Sets do not store duplicates.)

##### ❓ Computing the Jaccard index

Complete the function `jaccard_index()` for the computation of the Jaccard index of two sets `a` and `b`.

In [ ]:
def jaccard_index(a: set, b: set) -> float:
    """Compute the Jaccard index of two sets."""
    if not a and not b:
        return 0.0

    # TODO: return the intersection size divided by the union size.
    pass  # Remove after implementing the TODO.

    # --- Solution (uncomment to check) ---
    # return len(a & b) / len(a | b)


In [ ]:
test_strings = [
    ("", ""),
    ("", "A"),
    ("A", "A"),
    ("A", "C"),
    ("AC", "GT"),
    ("ACG", "GT"),
    ("ACG", "CGT"),
]

# Transform the list into a list of tuples of sets.
test_sets = [(set(item1), set(item2)) for item1, item2 in test_strings]

# Compute and print the Jaccard indexes.
for set_a, set_b in test_sets:
    print(f"J({set_a}, {set_b}) = {jaccard_index(a=set_a, b=set_b)}")

### ❓ Exercise 5 — Compare one homologous gene in human and chimpanzee
**Task:** Locate the **pyruvate kinase** sequence descriptions in the human and chimpanzee FASTA dictionaries, extract both sequences, build their 10-mer **sets**, and compute the Jaccard index.

**Hints:** First inspect `read_fasta_file(...).keys()` and search for `pyruvate kinase` case-insensitively. Do not assume the full FASTA header is exactly `pyruvate kinase`. Convert the first output of `build_kmers(...)` to a `set` before computing Jaccard.

**What this teaches:** Turning a real biological comparison into reusable Python operations.

**Expected outcome:** The final `print(f"J = {j}")` reports a number between 0 and 1. The exact value depends on the provided FASTA sequences; it is **not** hardcoded here.

**Interpretation prompt:** Does a high k-mer Jaccard imply that the DNA sequences are identical? Why not?

##### ❓ Similarity of homologous genes

Now, let's compare the similarity of homologous genes in different species:

1. Compute all 10-mers for the DNA sequence of the pyruvate kinase gene in _H. sapiens_ and _P. troglodytes_.
2. Then, compute the Jaccard index between the two 10-mer sets.

In [ ]:
# TODO: find the pyruvate kinase entries, get their DNA sequences,
#       create 10-mer sets, and calculate j.
# YOUR CODE

# --- Solution (uncomment to check) ---
# human = read_fasta_file(file_path="data/homologous-genes/h-sapiens.fasta")
# chimp = read_fasta_file(file_path="data/homologous-genes/p-troglodytes.fasta")
# human_key = next(key for key in human if "pyruvate kinase" in key.lower())
# chimp_key = next(key for key in chimp if "pyruvate kinase" in key.lower())
# human_kmers = set(build_kmers(human[human_key], 10)[0])
# chimp_kmers = set(build_kmers(chimp[chimp_key], 10)[0])
# j = jaccard_index(human_kmers, chimp_kmers)

print(f"J = {j}")


### ❓ Exercise 6 — Compare all genes and species pairs
**Task:** For each homologous gene and each ordered pair of species, compute a k-mer-set Jaccard score. Store it in `similarity_matrix[gene_index, species_index_1, species_index_2]`. Plot one heatmap per gene using the original plotting commands.

**Hints:** Use three nested loops: `(gene_index, gene_name)`, `(species_index_1, species_1)`, and `(species_index_2, species_2)`. Create sets with `set(build_kmers(sequence, current_k)[0])`. The input dictionaries are keyed by FASTA descriptions, so check that corresponding gene labels are present for every species.

**What this teaches:** Data organization, nested loops, pairwise comparisons, and interpreting similarity heatmaps.

**Expected outcome:** The array has shape `(n_genes, 5, 5)` for this dataset. Every heatmap should be symmetric; its diagonal is typically 1.0 for nonempty k-mer sets. Each heatmap is one homologous gene across five species.

**Experiment:** Try `current_k = 4`, `6`, and `10`. How does the apparent similarity change, and why? Smaller `k` has fewer possible distinct words and may saturate similarity.

##### ❓ Comparison of homologous genes across species

Now, iterate through all genes and compute the Jaccard indexes between all possible species-species pairs of _k_-mer sets.

Store the computed Jaccard indexes in a similarity array with the dimensions `shape=[n_genes, n_species, n_species]`.

Use varying values for $k$.
(A good starting point is $k = 4$.)

In [ ]:
import numpy as np

current_k = 4

# Read in all gene sequences for all species.
file_list = list_files_in_directory(directory="data/homologous-genes/")
all_sequences = {}
for file in file_list:
    species = os.path.splitext(os.path.basename(file))[0]
    sequences = read_fasta_file(file_path=file)
    all_sequences[species] = sequences

# Get all gene names.
gene_names = list(all_sequences[list(all_sequences.keys())[0]].keys())
n_genes = len(gene_names)

# Get all species.
speciess = list(all_sequences.keys())
n_species = len(speciess)

# Set up the similarity matrix.
similarity_matrix = np.zeros(shape=[n_genes, n_species, n_species])

# TODO: Fill the matrix using nested loops. Then plot a heatmap per gene.
# YOUR CODE

# --- Solution (uncomment to check) ---
# for i, gene_name in enumerate(gene_names):
#     for row, species_a in enumerate(speciess):
#         for col, species_b in enumerate(speciess):
#             # The original files use corresponding gene descriptions.
#             kmers_a = set(build_kmers(all_sequences[species_a][gene_name], current_k)[0])
#             kmers_b = set(build_kmers(all_sequences[species_b][gene_name], current_k)[0])
#             similarity_matrix[i, row, col] = jaccard_index(kmers_a, kmers_b)
#
#     plt.imshow(X=similarity_matrix[i], vmin=0, vmax=1)
#     plt.title(label=f"{gene_name}")
#     plt.xticks(ticks=range(len(speciess)), labels=speciess, rotation=90)
#     plt.yticks(ticks=range(len(speciess)), labels=speciess)
#     plt.colorbar(label="Jaccard similarity")
#     plt.show()


## Wrap-up — Reflect on what you learned
1. **FASTA parsing:** What are the dictionary keys and values?
2. **k-mers:** Why are k-mers overlapping? How many do you expect for sequence length `L` and a valid `k`?
3. **Spectrum:** How does “frequency of a k-mer” differ from “number of k-mers at that frequency”?
4. **Jaccard:** Why do repeated occurrences disappear when we use sets?
5. **Species comparison:** What information is lost when whole sequences are reduced to sets of k-mers?

**Optional challenge:** Verify that `similarity_matrix[i]` is symmetric using `np.allclose(similarity_matrix[i], similarity_matrix[i].T)`. Investigate the diagonal and compare multiple values of `k`.

**Take-home message:** K-mers turn DNA into countable and comparable features. They enable simple, alignment-free exploration of sequences, but are sensitive to the choice of `k` and do not preserve full sequence order.